# To sample more data at the drop region of reliability vs h_dist

In [3]:
'''Different way of sampling the drop region, by finding the difference in reliability over a rolling window of 3'''

import pandas as pd
import numpy as np

# Implement a function to calculate the difference in reliability column of a dataframe over a rolling window
def get_rolling_drop_region(df, window, threshold):
    # Calculate the difference in reliability over a rolling window.
    df["Rolling_Diff"] = df["Reliability"].rolling(window=window, center=True).apply(lambda x: x.iloc[0] - x.iloc[-1], raw=False)
    # The drop region is the region where the rolling difference exceeds a certain threshold
    drop_region = df[(df["Rolling_Diff"] > threshold)]
    if drop_region.empty:
        return None
    else:
        return (drop_region["Horizontal_Distance"].min(), drop_region["Horizontal_Distance"].max())

def merge_regions(region1, region2, region3):
    """
    Merge up to three regions, returning a list of non-overlapping regions.
    
    Args:
        region1, region2, region3: Each can be a tuple (min, max), (None, None), or None
    
    Returns:
        List of tuples representing non-overlapping regions, sorted by start point
    """
    # Collect non-None regions (treating (None, None) as None)
    regions = []
    for region in [region1, region2, region3]:
        if region is not None and region != (None, None):
            regions.append(region)
    
    # If no regions, return empty list
    if not regions:
        return []
    
    # Sort regions by their start point
    regions.sort(key=lambda x: x[0])
    
    # Merge overlapping regions
    merged = [regions[0]]
    
    for current_min, current_max in regions[1:]:
        last_min, last_max = merged[-1]
        
        # Check if current region overlaps with the last merged region
        if current_min <= last_max:
            # Overlapping or adjacent - merge them
            merged[-1] = (last_min, max(last_max, current_max))
        else:
            # Non-overlapping - add as separate region
            merged.append((current_min, current_max))
    
    return merged

height = [60, 90, 120, 150, 180, 210, 240, 270, 300]
usi = [10, 20, 66.7, 100]
bitrate = [39] # 6.5, 13, 19.5, 26, 39, 52, 58.5, 65
# NOTE: Doing this bitrate by bitrate

dl_df = pd.read_csv("/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_DJISpark/data_processed/Downlink_Reliability.csv")
ul_df = pd.read_csv("/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_DJISpark/data_processed/Uplink_Reliability.csv")
vid_df = pd.read_csv("/media/research-student/KingstonSSD/FANET_Dataset/Dataset_NP100000_DJISpark/data_processed/Video_Reliability.csv")

dl_df["Reliability"] = dl_df["Num_Reliable"] / dl_df["Num_Sent"]
ul_df["Reliability"] = ul_df["Num_Reliable"] / ul_df["Num_Sent"]
vid_df["Reliability"] = vid_df["Num_Reliable"] / vid_df["Num_Sent"]

height_list = []
usi_list = []
bitrate_list = []
hdist_list = []

# For each combination of height, usi, bitrate, find the drop region (the region where 0.05 < value < 0.95)
for h in height:
    for u in usi:
        for b in bitrate:
            # Extract the df for the current combination
            dl_subset = dl_df[(dl_df['Height'] == h) & (dl_df['UAV_Sending_Interval'] == u) & (dl_df['Bitrate'] == b)].reset_index(drop=True)
            ul_subset = ul_df[(ul_df['Height'] == h) & (ul_df['UAV_Sending_Interval'] == u) & (ul_df['Bitrate'] == b)].reset_index(drop=True)
            vid_subset = vid_df[(vid_df['Height'] == h) & (vid_df['UAV_Sending_Interval'] == u) & (vid_df['Bitrate'] == b)].reset_index(drop=True)

            # Check for sharp drops in reliability
            dl_drop_region = get_rolling_drop_region(dl_subset, window=3, threshold=0.1)
            ul_drop_region = get_rolling_drop_region(ul_subset, window=3, threshold=0.1)
            vid_drop_region = get_rolling_drop_region(vid_subset, window=3, threshold=0.1)

            # For links with sharp drop, find the region where reliability drops from >0.95 to <0.05
            if dl_drop_region != None:
                dl_start_hdist, dl_end_hdist = dl_drop_region
            else:
                dl_start_hdist = None
                dl_end_hdist = None

            if ul_drop_region != None:
                ul_start_hdist, ul_end_hdist = ul_drop_region
            else:
                ul_start_hdist = None
                ul_end_hdist = None

            if vid_drop_region != None:
                vid_start_hdist, vid_end_hdist = vid_drop_region
            else:
                vid_start_hdist = None
                vid_end_hdist = None

            # For the links with sharp drops, find the union of the drop regions
            drop_regions = merge_regions(
                (dl_start_hdist, dl_end_hdist) if dl_drop_region else None,
                (ul_start_hdist, ul_end_hdist) if ul_drop_region else None,
                (vid_start_hdist, vid_end_hdist) if vid_drop_region else None
            )

            if not drop_regions:
                print(f"Skipping for Height: {h}, USI: {u}, Bitrate: {b} (no sharp drop detected)")
                continue

            for region in drop_regions:
                start_hdist, end_hdist = region
                sample_hdist = [i for i in range(int(start_hdist), int(end_hdist)+1)]
                hdist_list = hdist_list + sample_hdist
                height_list = height_list + [h]*len(sample_hdist)
                usi_list = usi_list + [u]*len(sample_hdist)
                bitrate_list = bitrate_list + [b]*len(sample_hdist)

                if (end_hdist - start_hdist) > 100:
                    print(f"Warning: Long drop region detected! Range: {end_hdist - start_hdist}, Height: {h}, USI: {u}, Bitrate: {b}")


In [4]:
# Print the lists so that we can copy it into the ini file
# Since this is a long list, print with 100 items per line
def print_list_in_lines(lst, items_per_line=100):
    for i in range(0, len(lst), items_per_line):
        print(', '.join(str(x) for x in lst[i:i+items_per_line]) + ', \\') # Print without the square brackets

# Print the first half of the lists
index = len(height_list) 
print("Horizontal Distance List:")
print_list_in_lines(hdist_list[:index])
print("Height List:")
print_list_in_lines(height_list[:index])
print("USI List:")
print_list_in_lines(usi_list[:index])
# print("Bitrate List:")
# print_list_in_lines(bitrate_list[:index])


Horizontal Distance List:
130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, \
189, 190, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 14